In [1]:
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "input.txt"
)

('input.txt', <http.client.HTTPMessage at 0x79e548269760>)

## Setup & Data Loading

Notes from KV cache implementation:
- During inference (after prefill), only one new token arrives each step
- Q is computed from only the new token (size 1)
- K and V can be concatenated onto whatever you already cached
- Q @ K^T still works if Q has shape (B, 1, hs) and K has shape (B, T, hs) → result is (B, 1, T)
- We don't need the causal mask during decode because we are only considering the most recent token

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import time
from dataclasses import dataclass, field
from typing import List, Dict, Tuple

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss, _ = model(X, Y)  # unpack 3 return values now
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

## The Global Block Cache

A prompt of 12 tokens becomes 3 blocks:

```text
Block 0: tokens[0:4]   → KV for positions 0, 1, 2, 3
Block 1: tokens[4:8]   → KV for positions 4, 5, 6, 7
Block 2: tokens[8:12]  → KV for positions 8, 9, 10, 11

```
Each block stores a fixed-size KV chunk: (1, BLOCK_SIZE, head_size) per (layer, head). Only full blocks (exactly BLOCK_SIZE tokens) are eligible for caching. The trailing partial block is never cached — it changes with every new decode token.

Question to ask yourself: Why can't you cache partial blocks?

Answer: because the block's content isn't finalized. During decode, new tokens append to the last block until it fills up. Only then is its token content fixed and its hash meaningful.

You need a global cache that lives outside any individual request — a shared pool that multiple requests can read from.

In [3]:
import hashlib

NONE_HASH = b'\x00' * 16  # sentinel for the first block (no parent)

def hash_block_tokens(parent_hash, token_ids):
    """Compute a chained content hash for a KV block."""
    data = (parent_hash, tuple(token_ids))
    return hashlib.md5(str(data).encode()).digest()

@dataclass
class CachedBlock:
    """A cached KV block with its content hash."""
    block_hash: bytes
    token_ids: tuple                    # the tokens this block covers
    kv_data: Dict[Tuple[int, int], Tuple[torch.Tensor, torch.Tensor]]
    # kv_data[(layer, head)] = (k, v), each (1, BLOCK_SIZE, head_size)
    last_access_step: int = 0          # for LRU eviction

class BlockCache:
    def __init__(self, max_blocks=64):
        self.max_blocks = max_blocks
        self.cache: Dict[bytes, CachedBlock] = {}  # hash → CachedBlock
        self.current_step = 0

    def lookup(self, block_hash) -> CachedBlock | None:
        """Look up a block by its content hash."""
        block = self.cache.get(block_hash)
        if block is not None:
            block.last_access_step = self.current_step  # touch for LRU
        return block

    def insert(self, block_hash, token_ids, kv_data):
        """Insert a completed block into the cache."""
        if len(self.cache) >= self.max_blocks:
            self._evict_lru()
            
        self.cache[block_hash] = CachedBlock(
            block_hash=block_hash,
            token_ids=token_ids,
            kv_data=kv_data,
        )

    def _evict_lru(self):
        """Evict the least-recently-used block."""
        oldest = min(self.cache.values(), key=lambda b: b.last_access_step)
        del self.cache[oldest.block_hash]

## Finding Cache Hits During Admission

When a new request arrives, the scheduler needs to figure out how many of its prompt tokens are already cached. This is done by computing block hashes from the prompt and checking each one against the BlockCache:

In [4]:
def find_cached_prefix(block_cache: BlockCache, prompt_tokens, block_size):
    """
        Walk the prompt left-to-right in block-sized chunks.
        Return the number of tokens that are fully cached
    """

    num_cached = 0
    parent_hash = NONE_HASH

    for start in range(0, len(prompt_tokens), block_size):
        end = start + block_size
        if end > len(prompt_tokens): break

        chunk = prompt_tokens[start:end]
        chunk_hash = hash_block_tokens(parent_hash, chunk)

        cached_block = block_cache.lookup(chunk_hash)

        if cached_block is None: break

        num_cached += block_size
        parent_hash = chunk_hash
    
    return num_cached

## Loading Cached KV Data onto the Request

When you find cached blocks, you need to reconstruct the request's KV cache from the cached data before running the prefill of the remaining suffix:

In [5]:
def load_cached_blocks(request, block_cache, prompt_tokens, block_size):
    """ 
    Load cached KV blocks onto a request and return how many tokens were cached. 
    Sets request.prefill_cursor to skip past the cached potion
    """

    parent_hash = NONE_HASH
    num_cached = 0

    for start in range(0, len(prompt_tokens), block_size):
        end = start + block_size
        if end > len(prompt_tokens): break

        chunk = prompt_tokens[start:end]
        chunk_hash = hash_block_tokens(parent_hash, chunk)
        cached = block_cache.lookup(chunk_hash) # returns block. 

        if cached is None: break

        for (layer, head), (k, v) in cached.kv_data.items():
            if (layer, head) in request.kv_cache:
                existing_k, existing_v = request.kv_cache[(layer, head)]

                request.kv_cache[(layer, head)] = (
                    torch.cat([existing_k, k.clone()], dim=1),
                    torch.cat([existing_v, v.clone()], dim=1)
                )
            else:
                request.kv_cache[(layer, head)] = (k.clone(), v.clone())

        num_cached += block_size
        parent_hash = chunk_hash
    
    request.prefill_cursor = num_cached

    return num_cached

## Request Dataclass

Each in-flight generation carries its own state:
- `prompt_tokens` — the initial context
- `max_new_tokens` — individual stopping condition (request 0 may want 20, request 1 may want 100)
- `generated_tokens` — accumulates one token per scheduler step
- `status` — lets the scheduler know whether to batch this request
- `kv_cache` — **per-request** KV cache, keyed by `(layer_idx, head_idx)`

If request 0 finishes after 20 tokens but request 1 needs 100, request 0 is evicted from the batch (its row disappears), and request 1 continues generating.

In [6]:
@dataclass
class Request:
    """Each in-flight generation carries its own state and KV cache."""
    id: int
    prompt_tokens: List[int]          # the original encoded prompt
    max_new_tokens: int               # how many tokens this request wants
    generated_tokens: List[int] = field(default_factory=list)
    status: str = "waiting"           # "waiting" -> "prefilling" -> "active" -> "done"
    prefill_cursor: int = 0
    _committed_blocks: int = 0

    # Hint 2: Per-request KV cache, keyed by (layer_idx, head_idx)
    # Each value is a (key_tensor, value_tensor) tuple of shape (1, T_i, head_size)
    # T_i grows by 1 each decode step — different requests have different T_i
    kv_cache: Dict[Tuple[int, int], Tuple[torch.Tensor, torch.Tensor]] = field(
        default_factory=dict
    )

    @property
    def tokens_so_far(self) -> List[int]:
        """Full sequence: prompt + everything generated."""
        return self.prompt_tokens + self.generated_tokens

    @property
    def num_generated(self) -> int:
        return len(self.generated_tokens)

    @property
    def is_done(self) -> bool:
        return self.num_generated >= self.max_new_tokens
    
    @property
    def is_fully_prefilled(self) -> bool:
        return self.prefill_cursor == len(self.prompt_tokens)

    def clear_cache(self):
        self.kv_cache.clear()

In [7]:
class Scheduler:
    def __init__(self, policy="fcfs", max_batch_size=4, token_budget=16, max_kv_tokens=22, block_size=4):
        self.policy = policy
        self.max_batch_size = max_batch_size
        self.token_budget = token_budget
        self.max_kv_tokens = max_kv_tokens
        self.block_size = block_size
        self.block_cache = BlockCache()

        self.waiting = []
        self.prefilling = []
        self.active = []
        self.preempted = []

    def promote(self, req):
        self.prefilling.remove(req)
        req.status = "active"
        self.active.append(req)
    
    def complete(self, req):
        self.active.remove(req)
        req.status = "done"

    def _sort_key(self, req):
        if self.policy == "fcfs":
            return (0, req.arrival_time)
        elif self.policy == "priority":
            return (req.priority, req.arrival_time)
    
    def add_request(self, req):
        key = self._sort_key(req)
        heapq.heappush(self.waiting, (*key, req.id, req))
    
    def is_done(self):
        return not (self.waiting or self.prefilling or self.active)
    
    def _maybe_admit(self, step):
        if self.prefilling:
            return
        
        if not self.waiting:
            return

        kv_used = sum(len(req.prompt_tokens) + req.num_generated for req in self.active + self.prefilling)

        _, _, _, candidate = self.waiting[0]

        num_cached = find_cached_prefix(self.block_cache, candidate.prompt_tokens, self.block_size)

        # print(f"[step {step}] Admitting req {req.id}: "
        #     f"{num_cached}/{len(req.prompt_tokens)} tokens cached "
        #     f"({num_cached // BLOCK_SIZE} blocks hit), "
        #     f"{len(req.prompt_tokens) - num_cached} tokens to prefill")


        actual_kv_cost = len(candidate.prompt_tokens) - num_cached

        if kv_used + actual_kv_cost > self.max_kv_tokens:
            return
        
        if len(self.active) + len(self.prefilling) >= self.max_batch_size: return

        heapq.heappop(self.waiting)
        load_cached_blocks(candidate, self.block_cache, candidate.prompt_tokens, self.block_size)
        candidate.arrival_time = step
        candidate.status = "prefilling"
        self.prefilling.append(candidate)
    
    def _maybe_preempt(self):
        kv_used = sum(len(req.prompt_tokens) + req.num_generated for req in self.active + self.prefilling)

        while self.active and kv_used > self.max_kv_tokens:
            victim = max(self.active, key=lambda r: (r.priority, -r.arrival_time))
            self.active.remove(victim)
            victim.clear_cache()
            victim.prefill_cursor = 0
            victim.generated_tokens = []
            victim.status = "waiting"
            self.preempted.append(victim)

            key = self._sort_key(victim)
            heapq.heappush(self.waiting, (*key, victim.id, victim))
            kv_used = sum(len(req.prompt_tokens) + req.num_generated for req in self.active + self.prefilling)

    def schedule(self, step: int):
        """
        Returns:
            prefill_req:  Request | None  — one request getting a prefill chunk (or None)
            decode_reqs:  List[Request]   — all requests currently being decoded (active)

        """
        self.block_cache.current_step = step

        self._maybe_admit(step)       # promote waiting → prefilling if memory allows
        self._maybe_preempt()         # evict if over memory budget

        prefill_req = self.prefilling[0] if self.prefilling else None
        decode_reqs = list(self.active)

        return prefill_req, decode_reqs


## Hint 2: Stateless Head — KV Cache Moved Outside the Model

**Before:** `Head` owned `self.key_cache` and `self.value_cache` — one monolithic `(B, T, hs)` tensor.
This breaks when different requests have different sequence lengths.

**After:** `Head` is stateless. The cache is:
1. Passed **into** `forward()` as `past_k, past_v`
2. Returned **out of** `forward()` as updated `(new_k, new_v)`
3. **Stored on the `Request` object**, keyed by `(layer_idx, head_idx)`

This threads through: `Head` → `MultiHeadAttention` → `Block` → `GPTLanguageModel`.

Also changed `nn.Sequential` → `nn.ModuleList` for `self.blocks` so we can pass
per-block cache into each block individually.

In [ ]:
class Head(nn.Module):
    """One head of self-attention — now STATELESS (no internal cache)."""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_k=None, past_v=None, attn_mask=None):
        """
        Args:
            x:      (B, T, C)       input embeddings
            past_k: (B, T_past, hs) cached keys, or None
            past_v: (B, T_past, hs) cached values, or None
        Returns:
            out:   (B, T, hs)           attention output
            new_k: (B, T_past+T, hs)    updated key cache   (None during training)
            new_v: (B, T_past+T, hs)    updated value cache  (None during training)
        """
        B, T, C = x.shape
        k = self.key(x)    # (B, T, hs)
        q = self.query(x)  # (B, T, hs)
        v = self.value(x)  # (B, T, hs)

        if not self.training:
            if past_k is not None:
                k = torch.cat([past_k, k], dim=1)
                v = torch.cat([past_v, v], dim=1)

            T_full = k.shape[1]

            wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5  # (B, T, T_full)

            causal_mask = torch.ones(T, T_full, device=x.device, dtype=torch.bool)

            if T > 1:
                new_token_mask = self.tril[:T, :T]
                causal_mask[:, -T:] = new_token_mask

            causal_mask = causal_mask.unsqueeze(0).expand(B, -1,- 1)
        
            if attn_mask is not None:
                new_valid = torch.ones(B, 1, T, device=x.device, dtype=torch.bool)
                full_pad_mask = torch.cat([attn_mask, new_valid], dim=-1)
                causal_mask = causal_mask & full_pad_mask

            wei = wei.masked_fill(~causal_mask, float("-inf"))
            wei = F.softmax(wei, dim=-1)
            wei = self.dropout(wei)
            out = wei @ v

            return out, k, v

        else:
            wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
            wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
            wei = F.softmax(wei, dim=-1)
            wei = self.dropout(wei)
            out = wei @ v
            return out, None, None            

class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention in parallel."""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, attn_mask=None):
        """
        Args:
            x:       (B, T, C)
            past_kv: list of (past_k, past_v) per head, or None
        Returns:
            out:    (B, T, n_embd)
            new_kv: list of (new_k, new_v) per head
        """
        if past_kv is None:
            past_kv = [(None, None)] * len(self.heads)

        outputs, new_kvs = [], []
        for i, h in enumerate(self.heads):
            pk, pv = past_kv[i]
            out, nk, nv = h(x, pk, pv, attn_mask=attn_mask)
            outputs.append(out)
            new_kvs.append((nk, nv))

        out = torch.cat(outputs, dim=-1)
        out = self.dropout(self.proj(out))
        return out, new_kvs


class FeedFoward(nn.Module):
    """A simple linear layer followed by a non-linearity."""

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer block: communication followed by computation."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x, past_kv=None, attn_mask=None):
        """
        Returns:
            x:      (B, T, n_embd)
            new_kv: list of (new_k, new_v) per head in this block
        """
        sa_out, new_kv = self.sa(self.ln1(x), past_kv, attn_mask=attn_mask)
        x = x + sa_out
        x = x + self.ffwd(self.ln2(x))
        return x, new_kv


class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # ModuleList instead of Sequential so we can pass per-block cache
        self.blocks = nn.ModuleList([Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, pos=None, past_kvs=None, attn_mask=None):
        """
        Args:
            idx:      (B, T) token indices
            targets:  (B, T) target indices, or None
            pos:      (B, T) explicit position indices, or None (uses arange)
            past_kvs: list-of-lists cache structure, or None
                      past_kvs[layer][head] = (key_tensor, value_tensor)
        Returns:
            logits:   (B, T, vocab_size)
            loss:     scalar or None
            new_kvs:  updated cache with same structure as past_kvs
        """
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)  # (B, T, C)

        if pos is None:
            pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # (T, C)
        else:
            pos_emb = self.position_embedding_table(pos)  # (B, T, C)

        x = tok_emb + pos_emb  # (B, T, C)

        # Thread cache through each block
        if past_kvs is None:
            past_kvs = [None] * len(self.blocks)

        new_kvs = []
        for i, block in enumerate(self.blocks):
            x, block_kv = block(x, past_kvs[i], attn_mask=attn_mask)
            new_kvs.append(block_kv)

        x = self.ln_f(x)          # (B, T, C)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss, new_kvs

    def generate(self, idx, max_new_tokens):
        """Original generate (no cache, full recompute) for reference."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## Training

Training is **unchanged** — during `model.train()`, every `Head` takes the training branch
and returns `(out, None, None)` for the cache. The cache is simply discarded via `_`.

In [9]:
model = GPTLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss, _ = model(xb, yb)  # _ discards the cache during training
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Quick sanity check with the original no-cache generate
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=200)[0].tolist()))

0.209729 M parameters
step 0: train loss 4.1959, val loss 4.1962
step 100: train loss 2.6229, val loss 2.6166
step 200: train loss 2.4555, val loss 2.4488
step 300: train loss 2.3810, val loss 2.3928
step 400: train loss 2.3202, val loss 2.3223
step 500: train loss 2.2364, val loss 2.2541
step 600: train loss 2.1812, val loss 2.2234
step 700: train loss 2.1326, val loss 2.1583
step 800: train loss 2.0932, val loss 2.1352
step 900: train loss 2.0499, val loss 2.0987
step 1000: train loss 2.0349, val loss 2.0819
step 1100: train loss 1.9994, val loss 2.0706
step 1200: train loss 1.9857, val loss 2.0726
step 1300: train loss 1.9638, val loss 2.0401
step 1400: train loss 1.9386, val loss 2.0296
step 1500: train loss 1.9028, val loss 1.9947
step 1600: train loss 1.8821, val loss 1.9914
step 1700: train loss 1.8785, val loss 1.9813
step 1800: train loss 1.8746, val loss 1.9878
step 1900: train loss 1.8451, val loss 1.9599
step 2000: train loss 1.8294, val loss 1.9525
step 2100: train loss 1.

## The Scheduler Loop

Now that each request owns its own KV cache, the next step is the **scheduler loop**:

```
while there are active requests OR the waiting queue is non-empty:
    1. Check the waiting queue — can any new requests join the batch?
    2. Build the input tensor from ALL active requests (each contributes 1 token)
    3. Forward pass → get logits for all active requests at once
    4. Sample next token for each request
    5. Check: did any request hit max_new_tokens? → remove it, emit result
    6. Go to 1
```

The key challenge will be **padding the KV caches** to a common T dimension
when batching multiple requests (since they have different sequence lengths).
You'll need to `torch.cat` along dim=0 after padding along dim=1.

After un-batching the results, scatter the updated caches back to each request.

In [ ]:
def assemble_batch_cache(requests):
    """
    Gather per-request KV caches into batched tensors.
    LEFT-pads shorter caches so new tokens always land at the right edge.

    Big problem: You have 3 active requests. Each owns its own KV cache. You need to feed them to the model as one
    batched tensor. But their caches have different lengths:

    Returns:
        past_kvs:    batched cache structure  [layer][head] = (B, T_max, hs)
        attn_mask:   (B, 1, T_max) bool — True = valid, False = padding
        pad_lengths: list of int — how many pad positions per request (for disassembly)
    """

    B = len(requests)
    lengths = [req.kv_cache[(0, 0)][0].shape[1] for req in requests]
    max_t = max(lengths)

    pad_lengths = [max_t - t for t in lengths] # pad lengths for every position in t

    attn_mask = torch.zeros(B, 1, max_t, device=device, dtype=torch.bool)

    for i, pad in enumerate(pad_lengths):
        attn_mask[i, 0, pad:] = True

    past_kvs = []

    for layer_idx in range(n_layer):
        block_kv = []

        for head_idx in range(n_head):
            keys, values = [], []

            for i, req in enumerate(requests):
                k, v = req.kv_cache[(layer_idx, head_idx)]
                if pad_lengths[i] > 0:
                    hs = k.shape[2]
                    pad = torch.zeros(1, pad_lengths[i], hs, device=device)
                    k = torch.cat([pad, k], dim=1)
                    v = torch.cat([pad, v], dim=1)

                keys.append(k)
                values.append(v)

            block_kv.append((torch.cat(keys, dim=0), torch.cat(values, dim=0)))

        past_kvs.append(block_kv)

    return past_kvs, attn_mask, pad_lengths

def assemble_fused_batch(decode_reqs: List[Request], prefill_req, chunk_size):
    """
    Build a single (B, T_max) input tensor + batched cache for the fused forward pass.

    Args:
        decode_reqs:  list of active Request objects (each contributes 1 token)
        prefill_req:  the request being prefilled (contributes chunk_size tokens), or None
        chunk_size:   number of prefill tokens this step

    Returns:
        batch_tokens:   (B, T_max) input tensor
        batch_positions: (B, T_max) position indices
        past_kvs:       batched cache [layer][head] = (B, T_max_cache, hs)
        attn_mask:      (B, 1, T_max_cache) bool mask for cached positions
        pad_info:       dict with per-row metadata for disassembly
    """

    num_new_tokens = []
    all_reqs = []

    for req in decode_reqs:
        all_reqs.append(req)
        num_new_tokens.append(1)
    
    if prefill_req:
        all_reqs.append(prefill_req)
        num_new_tokens.append(chunk_size)

    B = len(all_reqs)
    T_max = max(num_new_tokens)

    batch_tokens = []
    batch_positions = []

    for req in decode_reqs:
        pos_val = len(req.tokens_so_far) - 1
        row = [0] * (T_max - 1) + [pos_val]
        batch_positions.append(row)

        token_row = [0] * (T_max - 1) + [req.tokens_so_far[-1]]
        batch_tokens.append(token_row)
    
    if prefill_req:
        cursor = prefill_req.prefill_cursor

        chunk_positions = list(range(cursor, cursor + chunk_size))

        padding = [0] * (T_max - chunk_size)

        batch_positions.append(padding + chunk_positions)

        chunk = prefill_req.prompt_tokens[cursor: cursor + chunk_size]
        pad = [0] * (T_max - chunk_size)

        batch_tokens.append(pad + chunk)

    batch_positions = torch.tensor(batch_positions, device=device)        
    batch_tokens = torch.tensor(batch_tokens, dtype=torch.long, device=device)  
    # Assemble KV cache 

    if prefill_req and not prefill_req.kv_cache:
        head_size = n_embd // n_head

        for li in range(n_layer):
            for hi in range(n_head):
                prefill_req.kv_cache[(li, hi)] = (
                    torch.empty(1, 0, head_size, device=device),
                    torch.empty(1, 0, head_size, device=device)
                )
        
    past_kvs, attn_mask, pad_lengths = assemble_batch_cache(all_reqs)
    
    return batch_tokens, batch_positions, past_kvs, attn_mask, pad_lengths

def disassemble_batch_cache(requests, new_kvs, pad_lengths):
    """
    Scatter batched KV cache back to per-request storage.
    After Head's torch.cat, each row is (T_max + 1) — strip the left-padding.
    """
    for layer_idx, block_kv in enumerate(new_kvs):
        for head_idx, (batched_k, batched_v) in enumerate(block_kv):
            for i, req in enumerate(requests):
                pad = pad_lengths[i]
                req.kv_cache[(layer_idx, head_idx)] = (
                    batched_k[i : i + 1, pad:, :],      # (1, T_i + 1, hs)
                    batched_v[i : i + 1, pad:, :],
                )

def disassemble_fused_cache(requests, new_kvs, num_new_tokens_per_req):
    for layer_idx, block_kv in enumerate(new_kvs):
        for head_idx, (batched_k, batched_v) in enumerate(block_kv):
            for i, req in enumerate(requests):

                t_new = num_new_tokens_per_req[i]

                k_new_valid = batched_k[i : i + 1, -t_new:, :]
                v_new_valid = batched_v[i : i + 1, -t_new:, :]

                if (layer_idx, head_idx) in req.kv_cache:
                    k_old, v_old = req.kv_cache[(layer_idx, head_idx)]
                    req.kv_cache[(layer_idx, head_idx)] = (
                        torch.cat([k_old, k_new_valid], dim=1),
                        torch.cat([v_old, v_new_valid], dim=1)
                    )
                else:
                    req.kv_cache[(layer_idx, head_idx)] = (k_new_valid, v_new_valid)

def commit_completed_blocks(request: Request, block_cache: BlockCache, block_size):
    """
    After a prefill step, check if any new full blocks were completed.
    If so, insert them into the global cache.
    """

    total_tokens = len(request.prompt_tokens) + request.num_generated
    num_full_blocks = request.prefill_cursor // block_size    

    parent_hash = NONE_HASH

    for block_idx in range(num_full_blocks):
        start = block_idx * block_size
        end = start + block_size

        chunk = request.prompt_tokens[start:end]
        block_hash = hash_block_tokens(parent_hash, chunk)

        if block_idx >= request._committed_blocks:
            kv_data = {}
            for (layer, head), (k, v) in request.kv_cache.items():
                kv_data[(layer, head)] = (
                    k[:, start:end, :].clone(),
                    v[:, start:end, :].clone()
                )

            block_cache.insert(block_hash, tuple(chunk), kv_data)

        parent_hash = block_hash
    
    request._committed_blocks = num_full_blocks

def build_tok_pos_kv(decode_reqs):
    batch_tokens = torch.cat([req._last_token for req in decode_reqs])

    batch_positions = torch.tensor([[len(req.tokens_so_far) - 1] for req in decode_reqs], device=device)

    past_kvs, attn_mask, pad_lengths = assemble_batch_cache(decode_reqs)

    return batch_tokens, batch_positions, past_kvs, attn_mask, pad_lengths

def scheduled_generate(model, requests, policy="fcfs", token_budget=16, max_kv_tokens=256):
    scheduler = Scheduler(policy, token_budget=token_budget, max_kv_tokens=max_kv_tokens)

    step = 0

    for req in requests:
        req.arrival_time = step
        scheduler.add_request(req)
    
    model.eval()

    with torch.no_grad():
        while not scheduler.is_done():

            prefill_req, decode_reqs = scheduler.schedule(step)

            if prefill_req:
            
                prefill_chunk_tokens = []
                
                remaining_budget = token_budget - len(scheduler.active)

                if remaining_budget > 0 and scheduler.prefilling:
                    p_req = scheduler.prefilling[0]

                    tokens_left = len(p_req.prompt_tokens) - p_req.prefill_cursor
                    chunk_size = min(remaining_budget, tokens_left)

                    chunk_start = p_req.prefill_cursor 

                    chunk_tokens = p_req.prompt_tokens[chunk_start: chunk_start + chunk_size]

                    prefill_chunk_tokens = torch.tensor([chunk_tokens], dtype=torch.long, device=device)

                    p_req.prefill_cursor += chunk_size

                if len(prefill_chunk_tokens) == 0 and not scheduler.active:
                    step += 1
                    continue

                if len(prefill_chunk_tokens) > 0:
                    pos = torch.arange(chunk_start, chunk_start + chunk_size, device=device).unsqueeze(0)

                    if p_req.kv_cache:
                        #  This format is wrong 
                        # logits, _, new_kvs = model(prefill_chunk_tokens, past_kvs=req.kv_cache)
                        # list[list[(k, v)]] is shape
                        
                        past_kvs = []
                        for layer_idx in range(n_layer):
                            block_kv = [(p_req.kv_cache[(layer_idx, hi)]) for hi in range(n_head)] 
                            past_kvs.append(block_kv)
                        
                        logits, _, new_kvs = model(prefill_chunk_tokens, pos=pos, past_kvs=past_kvs)

                    else:
                        logits, _, new_kvs = model(prefill_chunk_tokens, pos=pos)

                    for li, bkv in enumerate(new_kvs):
                        for hi, (k, v) in enumerate(bkv):
                            p_req.kv_cache[(li, hi)] = (k, v)


                    logits = logits[:, -1, :]
                    probs = F.softmax(logits, dim=-1)
                    idx_next = torch.multinomial(probs, num_samples=1)

                    if prefill_req.is_fully_prefilled:
                        prefill_req.generated_tokens.append(idx_next.item())
                        prefill_req._last_token = idx_next
                        commit_completed_blocks(prefill_req, scheduler.block_cache, scheduler.block_size)
                        # print(f"[step {step}] Committed {num_new_blocks} blocks from req {req.id} to cache "
                        #     f"(cache size: {len(block_cache.cache)}/{block_cache.max_blocks})")                        
                        scheduler.promote(prefill_req)

            if decode_reqs:
                B_active = len(scheduler.active)

                batch_tokens, batch_positions, past_kvs, attn_mask, pad_lengths = build_tok_pos_kv(decode_reqs)

                logits, _, new_kvs = model(
                    batch_tokens,
                    pos=batch_positions,
                    past_kvs=past_kvs,
                    attn_mask=attn_mask
                )

                logits = logits[:, -1, :]
                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1)

                disassemble_batch_cache(decode_reqs, new_kvs, pad_lengths)
                
                for i, req in enumerate(decode_reqs):
                    req.generated_tokens.append(idx_next[i].item())
                    req._last_token = idx_next[i : i + 1]
                
                for req in decode_reqs:
                    if req.is_done:
                        scheduler.complete(req)
            
            step += 1
        
        return scheduler

In [11]:
from numpy import remainder
## Interleave generate

def interleaved_generate(model, requests, policy="fcfs", token_budget=16, max_kv_tokens=256):
    scheduler = Scheduler(policy, token_budget=token_budget, max_kv_tokens=max_kv_tokens)

    step = 0

    for req in requests:
        req.arrival_time = step
        scheduler.add_request(req)

    model.eval()

    with torch.no_grad():
        while not scheduler.is_done():
            prefill_req, decode_reqs = scheduler.schedule(step)

            chunk_size = 0
            remaining_budget = token_budget - len(decode_reqs)

            if remaining_budget > 0 and prefill_req is not None:
                tokens_left = len(prefill_req.prompt_tokens) - prefill_req.prefill_cursor

                chunk_size = min(remaining_budget, tokens_left)

            if chunk_size == 0 and not decode_reqs:
                step += 1
                continue

            # 3. ── SINGLE FUSED MODEL CALL ──
            # Use your already-written helper to build the batched inputs
            batch_tokens, batch_positions, past_kvs, attn_mask, pad_lengths = assemble_fused_batch(
                decode_reqs, 
                prefill_req if chunk_size > 0 else None, 
                chunk_size
            )

            logits, _, new_kvs = model(
                batch_tokens,
                pos=batch_positions,
                past_kvs=past_kvs,
                attn_mask=attn_mask
            )

            ## DISASSEMBLY

            all_reqs = decode_reqs[:]
            num_new_tokens_per_req = [1] * len(decode_reqs)

            if chunk_size > 0:
                all_reqs.append(prefill_req)
                num_new_tokens_per_req.append(chunk_size)
                
            disassemble_fused_cache(all_reqs, new_kvs, num_new_tokens_per_req)
            
            # 5. ── POST-PROCESSING ──
            # Handle decode requests (they are the first N rows in the batch)

            if len(decode_reqs) > 0:
                logits_decode = logits[:len(decode_reqs), -1, :]
                probs = F.softmax(logits_decode, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1)

                for i, req in enumerate(decode_reqs):
                    req.generated_tokens.append(idx_next[i].item())
                    req._last_token = idx_next[i : i + 1]
                    if req.is_done:
                        scheduler.complete(req)
            
            if chunk_size > 0:
                prefill_req.prefill_cursor += chunk_size
            
                if prefill_req.is_fully_prefilled:
                    prefill_logits = logits[-1:, -1, :]
                    probs = F.softmax(prefill_logits, dim=-1)
                    idx_next = torch.multinomial(probs, num_samples=1)
                    
                    prefill_req.generated_tokens.append(idx_next.item())
                    prefill_req._last_token = idx_next
                    commit_completed_blocks(prefill_req, scheduler.block_cache, BLOCK_SIZE)
                    scheduler.promote(prefill_req)

            step += 1

    return scheduler                    


In [14]:
import heapq 

print("=" * 60)
print("Test 1: Regression — fused version produces correct output")
print("=" * 60)

BLOCK_SIZE = 4
torch.manual_seed(42)

reqs = [
    Request(id=0, prompt_tokens=encode("O Romeo, "),     max_new_tokens=15),
    Request(id=1, prompt_tokens=encode("To be or "),     max_new_tokens=15),
    Request(id=2, prompt_tokens=encode("KING HENRY:\n"), max_new_tokens=10),
]

s = interleaved_generate(model, reqs, policy="fcfs",
                         token_budget=16, max_kv_tokens=256)

for req in sorted(reqs, key=lambda r: r.id):
    print(f"  Req {req.id} ({req.status}, {req.num_generated} tokens): "
          f"'{decode(req.tokens_so_far)}'")
    assert req.status == "done", f"❌ Req {req.id} not done!"
    assert req.num_generated == req.max_new_tokens
    k, _ = req.kv_cache[(0, 0)]
    expected_T = len(req.prompt_tokens) + req.num_generated - 1
    assert k.shape[1] == expected_T, (
        f"❌ Req {req.id}: cache T={k.shape[1]}, expected {expected_T}"
    )

print("\n✅ Test 1 passed!")


Test 1: Regression — fused version produces correct output
  Req 0 (done, 15 tokens): 'O Romeo, dour affustend,'
  Req 1 (done, 15 tokens): 'To be or curn, stake you'
  Req 2 (done, 10 tokens): 'KING HENRY:
Mast now y'

✅ Test 1 passed!


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL: Test 2 — Single decode + single prefill, fused in one call
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("Test 2: Single decode + single prefill, fused in one call")
print("=" * 60)

BLOCK_SIZE = 4
torch.manual_seed(42)

# Req 0 has a short 3-token prompt → prefills instantly, then decodes.
# Req 1 arrives at step 0 with a 12-token prompt, token_budget=8.
#   Step 0: req 0 prefills (3 tokens) + first decode (1 token).
#           Req 1 starts prefilling. Budget: 8 - 1(decode for req0) = 7 tokens for req 1 prefill.
#   Step 1: req 0 decode (1 token) + req 1 remaining 5 prefill tokens = 6 ≤ 8.
#   Step 2: req 1 fully prefilled, joins decode. Both decoding.

reqs = [
    Request(id=0, prompt_tokens=encode("Hi!"),                    max_new_tokens=10),
    Request(id=1, prompt_tokens=encode("Good morrow!"),           max_new_tokens=10),
]

print(f"  Req 0 prompt: {len(reqs[0].prompt_tokens)} tokens")
print(f"  Req 1 prompt: {len(reqs[1].prompt_tokens)} tokens")
print(f"  Token budget: 8\n")

s = interleaved_generate(model, reqs, policy="fcfs",
                         token_budget=8, max_kv_tokens=256)

for req in sorted(reqs, key=lambda r: r.id):
    print(f"  Req {req.id} ({req.status}, {req.num_generated} tokens): "
          f"'{decode(req.tokens_so_far)}'")
    assert req.status == "done", f"Req {req.id} not done!"
    assert req.num_generated == req.max_new_tokens, (
        f"Req {req.id}: expected {req.max_new_tokens} tokens, got {req.num_generated}"
    )
    # Verify cache shape
    k, _ = req.kv_cache[(0, 0)]
    expected_T = len(req.prompt_tokens) + req.num_generated - 1
    assert k.shape[1] == expected_T, (
        f"Req {req.id}: cache T={k.shape[1]}, expected {expected_T}"
    )

print("\n✅ Test 2 passed — decode + prefill correctly fused in one forward pass!")


Test 2: Single decode + single prefill, fused in one call
  Req 0 prompt: 3 tokens
  Req 1 prompt: 12 tokens
  Token budget: 8

  Req 0 (done, 10 tokens): 'Hi!

BRICHAnd'
  Req 1 (done, 10 tokens): 'Good morrow!
That just'

✅ Test 2 passed — decode + prefill correctly fused in one forward pass!


In [15]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL: Test 3 — Forward pass count (single model call per step)
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("Test 3: Forward pass count — exactly 1 model() call per step")
print("=" * 60)

BLOCK_SIZE = 4
torch.manual_seed(42)

# Monkey-patch model.forward to count calls
call_count = 0
_orig_forward = model.forward

def counting_forward(*args, **kwargs):
    global call_count
    call_count += 1
    return _orig_forward(*args, **kwargs)

model.forward = counting_forward

reqs = [
    Request(id=0, prompt_tokens=encode("Hello"),                  max_new_tokens=8),
    Request(id=1, prompt_tokens=encode("Now is the winter of"),   max_new_tokens=5),
]

# Req 0: 5-token prompt, fully prefills in one step, then decodes.
# Req 1: 20-token prompt, arrives at step 0.
# With token_budget=10:
#   Each step should make exactly ONE model() call — the fused call.
#   (The two-call version would make 2 calls per step when both are active.)

call_count = 0
s = interleaved_generate(model, reqs, policy="fcfs",
                         token_budget=10, max_kv_tokens=256)

# Restore original forward
model.forward = _orig_forward

# Count how many total steps the scheduler ran
# (We can't easily get step count from outside, but we know the total
#  forward calls should equal the number of steps — one per step.)
total_tokens_generated = sum(r.num_generated for r in reqs)
print(f"  Total forward calls: {call_count}")
print(f"  Total tokens generated: {total_tokens_generated}")

for req in reqs:
    assert req.status == "done", f"Req {req.id} not done!"
    print(f"  Req {req.id}: {req.num_generated} tokens, '{decode(req.tokens_so_far)}'")

# The key assertion: with interleaving, each step should produce exactly 1 forward call.
# We can't know the exact step count without instrumenting the loop, but we CAN verify
# that call_count < what the two-call version would produce.
# As a sanity check: at minimum, every step with both prefill+decode active saves 1 call.
print(f"\n  (For comparison, two-call version would use more forward calls")
print(f"   during steps where both prefill and decode are active.)")

# Verify outputs are valid
for req in reqs:
    k, _ = req.kv_cache[(0, 0)]
    expected_T = len(req.prompt_tokens) + req.num_generated - 1
    assert k.shape[1] == expected_T, (
        f"Req {req.id}: cache T={k.shape[1]}, expected {expected_T}"
    )

# Now count what the two-call version would have used
call_count_two = 0
model.forward = counting_forward

torch.manual_seed(42)
reqs_two = [
    Request(id=0, prompt_tokens=encode("Hello"),                  max_new_tokens=8),
    Request(id=1, prompt_tokens=encode("Now is the winter of"),   max_new_tokens=5),
]
call_count = 0
_ = scheduled_generate(model, reqs_two, policy="fcfs",
                        token_budget=10, max_kv_tokens=256)
call_count_two = call_count
model.forward = _orig_forward

print(f"\n  Forward calls — interleaved: {call_count - call_count_two + call_count_two}")
# Recount properly
print(f"  Forward calls — two-call:    {call_count_two}")

# The interleaved version should have strictly fewer forward calls
# (Actually, let me restructure this to be cleaner)

# ── Clean recount ──
model.forward = counting_forward

# Count interleaved
torch.manual_seed(42)
reqs_a = [
    Request(id=0, prompt_tokens=encode("Hello"),                  max_new_tokens=8),
    Request(id=1, prompt_tokens=encode("Now is the winter of"),   max_new_tokens=5),
]
call_count = 0
_ = interleaved_generate(model, reqs_a, policy="fcfs", token_budget=10, max_kv_tokens=256)
calls_interleaved = call_count

# Count two-call
torch.manual_seed(42)
reqs_b = [
    Request(id=0, prompt_tokens=encode("Hello"),                  max_new_tokens=8),
    Request(id=1, prompt_tokens=encode("Now is the winter of"),   max_new_tokens=5),
]
call_count = 0
_ = scheduled_generate(model, reqs_b, policy="fcfs", token_budget=10, max_kv_tokens=256)
calls_two_call = call_count

model.forward = _orig_forward

print(f"\n  ── Final comparison ──")
print(f"  Interleaved forward calls: {calls_interleaved}")
print(f"  Two-call forward calls:    {calls_two_call}")
print(f"  Calls saved:               {calls_two_call - calls_interleaved}")

assert calls_interleaved < calls_two_call, (
    f"❌ Interleaved ({calls_interleaved}) should use fewer calls than "
    f"two-call ({calls_two_call})!"
)

print("\n✅ Test 3 passed — interleaved uses fewer forward calls!")


Test 3: Forward pass count — exactly 1 model() call per step
  Total forward calls: 8
  Total tokens generated: 13
  Req 0: 8 tokens, 'Hellowould af'
  Req 1: 5 tokens, 'Now is the winter of shal'

  (For comparison, two-call version would use more forward calls
   during steps where both prefill and decode are active.)

  Forward calls — interleaved: 11
  Forward calls — two-call:    11

  ── Final comparison ──
  Interleaved forward calls: 8
  Two-call forward calls:    11
  Calls saved:               3

✅ Test 3 passed — interleaved uses fewer forward calls!


In [16]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL: Test 4 — Budget arithmetic with many decode requests
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("Test 4: Budget arithmetic — 5 decode + 20-token prefill")
print("=" * 60)

BLOCK_SIZE = 4
torch.manual_seed(42)

# 5 short-prompt requests that prefill instantly and decode.
# 1 long-prompt request (20 tokens) that must be chunked around the decode budget.
# token_budget=10, so:
#   When 5 are decoding, remaining = 10 - 5 = 5 tokens/step for prefill.
#   20 tokens / 5 per step = 4 steps to fully prefill.

short_prompts = ["Hi", "Go", "Ye", "No", "Ok"]
reqs = []
for i, p in enumerate(short_prompts):
    reqs.append(Request(id=i, prompt_tokens=encode(p), max_new_tokens=10))

long_prompt = "KING RICHARD THE THI"  # exactly 20 chars = 20 tokens
reqs.append(Request(id=5, prompt_tokens=encode(long_prompt), max_new_tokens=5))

print(f"  Short requests: {len(short_prompts)} × 2-token prompts, 10 new tokens each")
print(f"  Long request:   {len(encode(long_prompt))}-token prompt, 5 new tokens")
print(f"  Token budget:   10\n")

s = interleaved_generate(model, reqs, policy="fcfs",
                         token_budget=10, max_kv_tokens=256)

for req in sorted(reqs, key=lambda r: r.id):
    status_icon = "✅" if req.status == "done" else "❌"
    print(f"  {status_icon} Req {req.id} ({req.status}, {req.num_generated} tokens): "
          f"'{decode(req.tokens_so_far)[:40]}...'")

# Verify all completed
for req in reqs:
    assert req.status == "done", f"Req {req.id} not done!"
    assert req.num_generated == req.max_new_tokens, (
        f"Req {req.id}: expected {req.max_new_tokens}, got {req.num_generated}"
    )
    # Verify the long request fully prefilled
    if req.id == 5:
        assert req.prefill_cursor == len(req.prompt_tokens), (
            f"Req 5: prefill_cursor={req.prefill_cursor}, "
            f"expected {len(req.prompt_tokens)}"
        )
    # Verify cache shape
    k, _ = req.kv_cache[(0, 0)]
    expected_T = len(req.prompt_tokens) + req.num_generated - 1
    assert k.shape[1] == expected_T, (
        f"Req {req.id}: cache T={k.shape[1]}, expected {expected_T}"
    )

print("\n✅ Test 4 passed — budget correctly split between decode and prefill!")


Test 4: Budget arithmetic — 5 decode + 20-token prefill
  Short requests: 5 × 2-token prompts, 10 new tokens each
  Long request:   20-token prompt, 5 new tokens
  Token budget:   10

  ✅ Req 0 (done, 10 tokens): 'Hintishful a...'
  ✅ Req 1 (done, 10 tokens): 'Go, shoulder...'
  ✅ Req 2 (done, 10 tokens): 'Ye lord, it ...'
  ✅ Req 3 (done, 10 tokens): 'Now whope an...'
  ✅ Req 4 (done, 10 tokens): 'Oknof my thu...'
  ✅ Req 5 (done, 5 tokens): 'KING RICHARD THE THIs: fo...'

✅ Test 4 passed — budget correctly split between decode and prefill!


In [17]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL: Test 5 — Budget fully consumed by decode (no prefill this step)
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("Test 5: Budget fully consumed by decode — prefill waits")
print("=" * 60)

BLOCK_SIZE = 4
torch.manual_seed(42)

# 4 short-prompt requests that prefill instantly.
# token_budget=4, so all 4 decode requests consume the entire budget.
# A 5th request with a long prompt must WAIT until a decode request finishes.
# When a decode request completes and frees a slot, the 5th request's prefill
# should resume.

reqs = [
    Request(id=0, prompt_tokens=encode("Hi"),   max_new_tokens=3),  # finishes quickly
    Request(id=1, prompt_tokens=encode("Go"),   max_new_tokens=3),
    Request(id=2, prompt_tokens=encode("Ye"),   max_new_tokens=3),
    Request(id=3, prompt_tokens=encode("No"),   max_new_tokens=3),
    # This one must wait — budget is 4, and 4 decode requests fill it
    Request(id=4, prompt_tokens=encode("Once upon a time"),  max_new_tokens=5),
]

print(f"  Requests 0-3: 2-token prompts, 3 new tokens each")
print(f"  Request 4:    {len(encode('Once upon a time'))}-token prompt, 5 new tokens")
print(f"  Token budget: 4\n")

s = interleaved_generate(model, reqs, policy="fcfs",
                         token_budget=4, max_kv_tokens=256)

for req in sorted(reqs, key=lambda r: r.id):
    status_icon = "✅" if req.status == "done" else "❌"
    print(f"  {status_icon} Req {req.id} ({req.status}, {req.num_generated} tokens): "
          f"'{decode(req.tokens_so_far)}'")

# Verify all completed — including req 4 which had to wait
for req in reqs:
    assert req.status == "done", f"Req {req.id} not done!"
    assert req.num_generated == req.max_new_tokens, (
        f"Req {req.id}: expected {req.max_new_tokens}, got {req.num_generated}"
    )
    # Verify cache shape
    k, _ = req.kv_cache[(0, 0)]
    expected_T = len(req.prompt_tokens) + req.num_generated - 1
    assert k.shape[1] == expected_T, (
        f"Req {req.id}: cache T={k.shape[1]}, expected {expected_T}"
    )

# Req 4 must have been fully prefilled eventually
assert reqs[4].prefill_cursor == len(reqs[4].prompt_tokens), (
    f"Req 4: prefill not completed (cursor={reqs[4].prefill_cursor})"
)

print("\n✅ Test 5 passed — prefill correctly waited and resumed when budget freed up!")


Test 5: Budget fully consumed by decode — prefill waits
  Requests 0-3: 2-token prompts, 3 new tokens each
  Request 4:    16-token prompt, 5 new tokens
  Token budget: 4

  ✅ Req 0 (done, 3 tokens): 'Hinti'
  ✅ Req 1 (done, 3 tokens): 'Go, a'
  ✅ Req 2 (done, 3 tokens): 'Ye sa'
  ✅ Req 3 (done, 3 tokens): 'Nownf'
  ✅ Req 4 (done, 5 tokens): 'Once upon a timents, '

✅ Test 5 passed — prefill correctly waited and resumed when budget freed up!


In [18]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL: Test 6 (Bonus) — Prefill-only step (no active decode requests)
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("Test 6: Prefill-only step — no active decode requests")
print("=" * 60)

BLOCK_SIZE = 4
torch.manual_seed(42)

# A single request with a long prompt. No decode requests exist yet.
# The fused batch degenerates to a pure prefill — verify this doesn't crash.

long_text = "Now is the winter of our discontent"  # 35 chars = 35 tokens
prompt = encode(long_text)
# Clamp to block_size to avoid position overflow
prompt = prompt[:block_size - 5]  # limit to 32 tokens

reqs = [
    Request(id=0, prompt_tokens=prompt, max_new_tokens=5),
]

print(f"  Prompt: {len(prompt)} tokens")
print(f"  Token budget: 8")
print(f"  Expected prefill chunks: ~{len(prompt) // 8} steps\n")

s = interleaved_generate(model, reqs, policy="fcfs",
                         token_budget=8, max_kv_tokens=256)

req = reqs[0]
print(f"  Status: {req.status}")
print(f"  Generated: {req.num_generated} tokens")
print(f"  Prefill cursor: {req.prefill_cursor}/{len(req.prompt_tokens)}")
print(f"  Output: '{decode(req.tokens_so_far)}'")

assert req.status == "done", f"Req not done!"
assert req.num_generated == 5
assert req.prefill_cursor == len(req.prompt_tokens)

k, _ = req.kv_cache[(0, 0)]
expected_T = len(req.prompt_tokens) + req.num_generated - 1
assert k.shape[1] == expected_T, (
    f"Cache T={k.shape[1]}, expected {expected_T}"
)

print("\n✅ Test 6 passed — prefill-only steps work correctly!")


Test 6: Prefill-only step — no active decode requests
  Prompt: 27 tokens
  Token budget: 8
  Expected prefill chunks: ~3 steps

  Status: done
  Generated: 5 tokens
  Prefill cursor: 27/27
  Output: 'Now is the winter of our did
Our'

✅ Test 6 passed — prefill-only steps work correctly!
